# Day 58 — Logistic Regression and the Sigmoid Function

This notebook covers Binary Classification Limitations of Linear Regression, The Sigmoid Activation Function ($\sigma(z)$), Decision Boundaries, and Scikit-Learn Logistic Regression Modeling

## Validation goals
- Analyze why linear regression fails for binary classification due to unbounded outputs exceeding valid probability ranges
- Define the sigmoid function and its mathematical mechanism for squashing unconstrained scores into probabilities between 0.0 and 1.0
- Establish classification decision boundaries using standard probability thresholds (0.5)
- Implement a binary classification pipeline in Python using Scikit-Learn, differentiating between hard labels (`predict`) and soft probabilities (`predict_proba`)

### 1. Why Linear Regression Fails for Classification

Imagine you want to predict whether an email is Spam (`1`) or Not Spam (`0`) based on how many suspicious keywords it contains.

If you try to draw a standard linear regression straight line through data points that only take values of 0 and 1:
- The best-fit line will tilt upward.
- However, because it's a straight line extending infinitely in both directions, it will predict outputs like $1.4$, $2.0$, or even $-0.5$.
- How do you interpret a prediction of "1.4 spam"? It doesn't make sense when you are trying to output a probability between 0% and 100%.

We need a mathematical function that takes any real-world input and "squeezes" or compresses it so that the output always falls strictly between $0.0$ and $1.0$. Enter the Sigmoid Function.

### 2. The Sigmoid Function: Squashing Output into Probabilities

### Concept Explanation

Logistic regression takes the standard linear equation ($z = \beta_0 + \beta_1x_1$) and passes it through a special activation function called the Sigmoid function (also known as the logistic function).

The mathematical formula for the sigmoid function is:$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
- Input ($z$): Any number from $-\infty$ to $+\infty$.
- Output ($\sigma(z)$): A strict decimal value between $0.0$ and $1.0$, which represents the estimated probability that the given data point belongs to the positive class (e.g., $P(\text{Spam} = 1 \vert{} X)$).

**Decision Boundaries**

Once the model outputs a probability (e.g., $0.78$), we apply a decision threshold (traditionally set at $0.5$):If probability $\ge 0.5$, classify as Class 1 (e.g., Spam).If probability $< 0.5$, classify as Class 0 (e.g., Not Spam).

### 3. Implementing Logistic Regression in Python (Scikit-Learn)

Let's write a clean Python script using Scikit-learn to train a logistic regression model on a binary classification task: predicting whether a student passes an exam based on the number of hours they studied.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# 1. Setup synthetic dataset: Hours Studied vs. Exam Result (0 = Fail, 1 = Pass)
np.random.seed(42)
study_hours = np.array([0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.75, 3.0, 3.5, 4.0, 4.5, 5.0]).reshape(-1, 1)
exam_results = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])  # Binary labels

# 2. Instantiate the Logistic Regression model
model = LogisticRegression()

# 3. Train (fit) the model on our data
model.fit(study_hours, exam_results)

# 4. Extract learned parameters
weight = model.coef_[0][0]
bias = model.intercept_[0]

print(f"Learned Weight (Coefficient): {weight:.4f}")
print(f"Learned Bias (Intercept):     {bias:.4f}\n")

# 5. Make probability predictions for new students (e.g., studying 1.8 hours vs. 3.2 hours)
new_students = np.array([[1.8], [3.2]])

# predict_proba returns probabilities for [Class 0, Class 1]
probabilities = model.predict_proba(new_students)
hard_predictions = model.predict(new_students)

for hours, prob, pred in zip(new_students, probabilities, hard_predictions):
    status = "Pass" if pred == 1 else "Fail"
    print(f"Studied {hours[0]} hours -> Probability of passing: {prob[1]:.2f} ({status})")

Learned Weight (Coefficient): 1.5532
Learned Bias (Intercept):     -3.7136

Studied 1.8 hours -> Probability of passing: 0.29 (Fail)
Studied 3.2 hours -> Probability of passing: 0.78 (Pass)


### Guided Question:

Even though it has "regression" in its name, why is logistic regression categorized as a classification algorithm rather than a regression algorithm?

### Answer:

Even though it shares a name with linear regression, logistic regression is fundamentally a classification algorithm because its ultimate goal is to predict discrete categories rather than continuous numbers.

The confusion stems from a naming quirk: logistic regression does use a regression equation under the hood, but it uses that equation to calculate the probability that an input belongs to a specific class.

**1. The Two-Step Mechanism: Linear Output to Probability**

Logistic regression splits its prediction process into two distinct stages:
- The Linear Step (The "Regression" Part): Just like standard linear regression, it calculates a weighted sum of the inputs plus a bias:$$z = \beta_0 + \beta_1x_1 + \dots + \beta_nx_n$$
However, this output ($z$) is an unconstrained real number ranging from $-\infty$ to $+\infty$. You cannot use an unbounded number to represent a discrete class.
- The Sigmoid Activation (The "Classification" Part): To fix this, the model passes that linear score through the sigmoid function (logistic function), which squashes any number into a strictly bounded range between 0 and 1:$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**2. Output Space: Probabilities vs. Continuous Values**

The defining difference between regression and classification lies in what the model returns:
- Linear Regression (Regression): Outputs an unconstrained continuous value (e.g., predicting a house price of $\$314,500$).
- Logistic Regression (Classification): Outputs a probability between 0 and 1 (e.g., predicting a 0.87 probability that an email is spam).

By applying a decision threshold (typically `0.5`), this continuous probability is converted into a discrete class label:
- If $\sigma(z) \ge 0.5 \rightarrow$ Class 1 (e.g., Spam)
- If $\sigma(z) < 0.5 \rightarrow$ Class 0 (e.g., Not Spam)

**3. Where Does the Word "Regression" Come From?**

Mathematically, the algorithm is performing regression, but it is doing so on the log-odds (the log of the odds ratio), not on the final class labels.

Instead of fitting a straight line directly to 0s and 1s (which fails because linear regression can predict values greater than 1 or less than 0, resulting in nonsensical probabilities), it fits a linear model to the logarithm of the odds of an event occurring. Because it models a continuous variable (log-odds) using linear techniques, statisticians named it "logistic regression."

**4. The Loss Function Differs**

Because logistic regression solves a classification task, using standard linear regression loss functions—like Mean Squared Error (MSE)—fails catastrophically.
- MSE on probabilities results in a non-convex cost function full of local minima, making gradient descent unreliable.
- Instead, logistic regression uses Binary Cross-Entropy (Log Loss), which heavily penalizes the model when it is confidently wrong (e.g., predicting a 0.99 probability for an instance that turns out to be class 0).

### Guided Question:

What is the primary purpose of passing a linear equation through the sigmoid function, and what range do its output values always fall into?

### Answer:

The primary purpose of passing a linear equation through the sigmoid function is to transform an unconstrained linear score into a valid probability that can be used for binary classification.

Here is a breakdown of its purpose and output range:

**1. The Primary Purpose**

A standard linear equation outputs a continuous score ($z$) that can range anywhere from $-\infty$ to $+\infty$. While useful for predicting quantities like house prices, an unbounded score cannot answer a classification question like "What is the chance this email is spam?"

The sigmoid function acts as a squashing function that converts that raw, unbounded score into an interpretable probability scale. This allows you to set a decision threshold (such as $0.5$) to cleanly separate classes (e.g., Spam vs. Not Spam).

**2. The Output Range**

The output values of the sigmoid function always fall strictly between 0 and 1 (exclusive of 0 and 1, mathematically expressed as the open interval $(0, 1)$).
- As the linear score $z$ approaches positive infinity ($+\infty$), the sigmoid output approaches 1.
- As the linear score $z$ approaches negative infinity ($-\infty$), the sigmoid output approaches 0.When $z = 0$, the sigmoid output evaluates to exactly 0.5.

### Guided Question:

In the Scikit-learn code snippet above, what is the functional difference between what model.predict() returns versus what model.predict_proba() returns?

### Answer:

In Scikit-learn's logistic regression, the primary difference is that `model.predict()` returns hard class labels (discrete decisions), while `model.predict_proba()` returns soft class probabilities (continuous confidence scores).

Here is a breakdown of how each method functions:

**1. `model.predict(X)` — Hard Predictions**

- What it returns: A 1D array of discrete class labels (in this binary case, either 0 or 1).
- How it works: It evaluates the underlying probabilities and applies a default decision threshold of 0.5.
    - If the calculated probability of the positive class (Class 1) is $\ge 0.5$, it outputs 1.
    - If it is $< 0.5$, it outputs 0.

**Use case:** When you need a final, definitive decision (e.g., classifying an email as strictly Spam or Not Spam).

**2. `model.predict_proba(X)` — Soft/Probabilistic Predictions**

- What it returns: A 2D array of floating-point probabilities with a shape of (`n_samples, n_classes`). For a binary classifier, each row contains two values: `[Probability of Class 0, Probability of Class 1] `(which always sum to `1.0`).
- How it works: It outputs the raw, unthresholded probability values derived directly from the sigmoid function.
- Use case: When you care about model certainty rather than just a binary outcome. For instance, knowing a student has a `0.52` probability of passing vs. a `0.99 `probability of passing provides critical context that a simple 1 or 0 label strips away.